# 🔍 Notebook 16: Database Queries — SQL ผ่าน Python

## สิ่งที่จะได้เรียนรู้
- SELECT, WHERE, ORDER BY
- INSERT, UPDATE, DELETE
- JOIN ตาราง
- GROUP BY และ Aggregate Functions
- ใช้ pandas กับ SQL queries
- Window Functions เบื้องต้น

---

> 🐳 ต้องรัน `docker compose up -d` ก่อน (ดู Notebook 15)

In [ ]:
import pandas as pd        # pd = ชื่อย่อของ pandas (ไลบรารีจัดการข้อมูล)
import numpy as np         # np = ชื่อย่อของ numpy (คำนวณตัวเลข)
import os                  # os = ไลบรารีจัดการไฟล์/โฟลเดอร์
from dotenv import load_dotenv  # โหลดค่าจากไฟล์ .env
from sqlalchemy import create_engine, text  # เชื่อมต่อ + เขียน SQL

# สร้างโฟลเดอร์ output สำหรับเก็บผลลัพธ์
os.makedirs("output", exist_ok=True)
# โหลดค่าตัวแปรจากไฟล์ .env (เก็บรหัสผ่านไว้ที่นี่)
load_dotenv("../.env")

# สร้าง Database URL จากค่าในไฟล์ .env
DATABASE_URL = "postgresql://{}:{}@{}:{}/{}".format(
    os.getenv("DB_USER", "icp_user"),          # ชื่อผู้ใช้
    os.getenv("DB_PASSWORD", "training_password"),  # รหัสผ่าน
    os.getenv("DB_HOST", "localhost"),          # ที่อยู่ server
    os.getenv("DB_PORT", "5432"),              # พอร์ต
    os.getenv("DB_NAME", "icp_training"),      # ชื่อ Database
)

# engine = ตัวเชื่อมต่อไปยัง Database
engine = create_engine(DATABASE_URL, echo=False)

# ทดสอบการเชื่อมต่อ
with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print(f"\u2705 Connected to PostgreSQL: {result.fetchone()[0][:30]}...")

## 1. สร้างตารางและข้อมูลตัวอย่าง

In [ ]:
# สร้างตาราง (drop ก่อนเพื่อเริ่มใหม่ทุกครั้ง)
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS test_results CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS test_orders CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS laboratories CASCADE"))
    conn.execute(text("DROP TABLE IF EXISTS elements CASCADE"))
    
    conn.execute(text("""
        CREATE TABLE elements (
            id SERIAL PRIMARY KEY,
            symbol TEXT NOT NULL UNIQUE,
            name TEXT NOT NULL,
            wavelength REAL,
            unit TEXT DEFAULT 'mg/kg'
        )
    """))
    
    conn.execute(text("""
        CREATE TABLE laboratories (
            id SERIAL PRIMARY KEY,
            code TEXT NOT NULL UNIQUE,
            name TEXT NOT NULL,
            location TEXT
        )
    """))
    
    conn.execute(text("""
        CREATE TABLE test_orders (
            id SERIAL PRIMARY KEY,
            order_id TEXT NOT NULL UNIQUE,
            sample_id TEXT,
            laboratory_id INTEGER REFERENCES laboratories(id),
            panel_name TEXT,
            order_date TEXT,
            analyst_name TEXT,
            status TEXT DEFAULT 'pending'
        )
    """))
    
    conn.execute(text("""
        CREATE TABLE test_results (
            id SERIAL PRIMARY KEY,
            order_id INTEGER REFERENCES test_orders(id),
            element_id INTEGER REFERENCES elements(id),
            concentration REAL,
            replicate INTEGER DEFAULT 1
        )
    """))
    conn.commit()

print("✅ Tables created")

In [ ]:
# เพิ่มข้อมูลตัวอย่าง
np.random.seed(42)

# Elements
elements = pd.DataFrame({
    "symbol": ["As", "Pb", "Cd", "Hg", "Cu", "Zn", "Fe", "Mn", "Cr"],
    "name": ["Arsenic", "Lead", "Cadmium", "Mercury", "Copper", "Zinc", "Iron", "Manganese", "Chromium"],
    "wavelength": [188.979, 220.353, 226.502, 194.168, 327.393, 206.200, 238.204, 257.610, 267.716],
    "unit": ["mg/kg"] * 9
})
elements.to_sql("elements", engine, if_exists="append", index=False)

# Laboratories
labs = pd.DataFrame({
    "code": ["L66", "L67", "L68"],
    "name": ["Lab 66 - Food Safety", "Lab 67 - Environmental", "Lab 68 - Pharma"],
    "location": ["Building A", "Building B", "Building C"]
})
labs.to_sql("laboratories", engine, if_exists="append", index=False)

# Test Orders (20 orders)
orders = pd.DataFrame({
    "order_id": [f"ORD-2026-{i:04d}" for i in range(1, 21)],
    "sample_id": [f"S-{np.random.randint(100, 999)}" for _ in range(20)],
    "laboratory_id": np.random.choice([1, 2, 3], 20),
    "panel_name": np.random.choice(["EU4", "EU9"], 20),
    "order_date": pd.date_range("2026-01-01", periods=20, freq="2D").astype(str),
    "analyst_name": np.random.choice(["Dr. Smith", "Dr. Jones", "Dr. Lee"], 20),
    "status": np.random.choice(["completed", "in_progress", "pending"], 20, p=[0.6, 0.2, 0.2]),
})
orders.to_sql("test_orders", engine, if_exists="append", index=False)

# Test Results (แต่ละ order มี 4 elements)
results_rows = []
for order_idx in range(1, 21):
    for elem_idx in range(1, 5):  # As, Pb, Cd, Hg
        results_rows.append({
            "order_id": order_idx,
            "element_id": elem_idx,
            "concentration": round(np.random.uniform(0.01, 5.0), 3),
            "replicate": 1,
        })
        results_rows.append({
            "order_id": order_idx,
            "element_id": elem_idx,
            "concentration": round(np.random.uniform(0.01, 5.0), 3),
            "replicate": 2,
        })

results = pd.DataFrame(results_rows)
results.to_sql("test_results", engine, if_exists="append", index=False)

print(f"✅ Sample data loaded:")
print(f"   Elements: {len(elements)}")
print(f"   Labs: {len(labs)}")
print(f"   Orders: {len(orders)}")
print(f"   Results: {len(results)}")

## 2. SELECT — อ่านข้อมูล

In [ ]:
# === SELECT ทั้งหมด ===
df = pd.read_sql("SELECT * FROM elements", engine)
df

In [ ]:
# === SELECT เฉพาะบางคอลัมน์ ===
df = pd.read_sql("SELECT symbol, name, wavelength FROM elements", engine)
df

In [ ]:
# === WHERE — กรองข้อมูล ===
df = pd.read_sql("""
    SELECT * FROM test_orders 
    WHERE status = 'completed'
    ORDER BY order_date DESC
""", engine)

print(f"Completed orders: {len(df)}")
df.head()

In [ ]:
# === WHERE กับหลายเงื่อนไข ===
df = pd.read_sql("""
    SELECT * FROM test_orders 
    WHERE status = 'completed' 
      AND analyst_name = 'Dr. Smith'
    ORDER BY order_date
""", engine)

print(f"Dr. Smith's completed orders: {len(df)}")
df

In [ ]:
# === LIKE — ค้นหาคำ ===
df = pd.read_sql("""
    SELECT * FROM test_orders
    WHERE order_id LIKE '%001%'
""", engine)
df

In [ ]:
# === IN — เลือกหลายค่า ===
df = pd.read_sql("""
    SELECT * FROM test_orders
    WHERE status IN ('pending', 'in_progress')
    ORDER BY status, order_date
""", engine)
df

## 3. JOIN — รวมข้อมูลจากหลายตาราง

```
test_orders  ←→  laboratories
     ↕
test_results ←→  elements
```

In [ ]:
# === INNER JOIN — Orders กับ Labs ===
df = pd.read_sql("""
    SELECT 
        o.order_id,
        o.sample_id,
        o.order_date,
        o.analyst_name,
        o.status,
        l.code AS lab_code,
        l.name AS lab_name
    FROM test_orders o
    INNER JOIN laboratories l ON o.laboratory_id = l.id
    ORDER BY o.order_date DESC
    LIMIT 10
""", engine)

print("Orders with Lab info:")
df

In [ ]:
# === JOIN หลายตาราง — ผลทดสอบแบบละเอียด ===
df = pd.read_sql("""
    SELECT 
        o.order_id,
        o.sample_id,
        l.code AS lab,
        e.symbol AS element,
        e.name AS element_name,
        r.concentration,
        r.replicate,
        o.order_date
    FROM test_results r
    INNER JOIN test_orders o ON r.order_id = o.id
    INNER JOIN elements e ON r.element_id = e.id
    INNER JOIN laboratories l ON o.laboratory_id = l.id
    WHERE o.order_id = 'ORD-2026-0001'
    ORDER BY e.symbol, r.replicate
""", engine)

print("Detailed results for ORD-2026-0001:")
df

## 4. GROUP BY — สรุปข้อมูล

In [ ]:
# === นับจำนวน orders ตาม status ===
df = pd.read_sql("""
    SELECT 
        status,
        COUNT(*) AS order_count
    FROM test_orders
    GROUP BY status
    ORDER BY order_count DESC
""", engine)

print("Orders by status:")
df

In [ ]:
# === นับจำนวน orders ตาม Lab และ Analyst ===
df = pd.read_sql("""
    SELECT 
        l.code AS lab,
        o.analyst_name,
        COUNT(*) AS order_count
    FROM test_orders o
    INNER JOIN laboratories l ON o.laboratory_id = l.id
    GROUP BY l.code, o.analyst_name
    ORDER BY lab, order_count DESC
""", engine)

print("Orders by Lab & Analyst:")
df

In [ ]:
# === Aggregate Functions (ฟังก์ชันรวมค่า): AVG, MIN, MAX, SUM ===
# pd.read_sql() = อ่านข้อมูลจาก SQL เข้ามาเป็น DataFrame
df = pd.read_sql("""
    SELECT 
        e.symbol,                                    -- สัญลักษณ์ธาตุ
        e.name,                                      -- ชื่อธาตุ
        COUNT(r.id) AS measurement_count,            -- นับจำนวนการวัด
        ROUND(AVG(r.concentration)::numeric, 3) AS avg_concentration,  -- ค่าเฉลี่ย
        ROUND(MIN(r.concentration)::numeric, 3) AS min_concentration,  -- ค่าต่ำสุด
        ROUND(MAX(r.concentration)::numeric, 3) AS max_concentration   -- ค่าสูงสุด
    FROM test_results r
    INNER JOIN elements e ON r.element_id = e.id     -- เชื่อมตาราง results กับ elements
    GROUP BY e.symbol, e.name                        -- จัดกลุ่มตามธาตุ
    ORDER BY avg_concentration DESC                  -- เรียงจากมากไปน้อย
""", engine)

print("Element Statistics:")
df

In [ ]:
# === HAVING — กรองผลหลัง GROUP BY ===
df = pd.read_sql("""
    SELECT 
        o.analyst_name,
        COUNT(*) AS order_count
    FROM test_orders o
    GROUP BY o.analyst_name
    HAVING COUNT(*) >= 5
    ORDER BY order_count DESC
""", engine)

print("Analysts with 5+ orders:")
df

## 5. INSERT, UPDATE, DELETE — แก้ไขข้อมูล

In [ ]:
# === INSERT — เพิ่มข้อมูล ===
with engine.connect() as conn:
    conn.execute(text("""
        INSERT INTO test_orders (order_id, sample_id, laboratory_id, panel_name, order_date, analyst_name, status)
        VALUES (:oid, :sid, :lid, :panel, :date, :analyst, :status)
    """), {
        "oid": "ORD-2026-0021",
        "sid": "S-NEW",
        "lid": 1,
        "panel": "EU4",
        "date": "2026-02-15",
        "analyst": "Dr. New",
        "status": "pending"
    })
    conn.commit()

# ตรวจสอบ
df = pd.read_sql("SELECT * FROM test_orders WHERE order_id = 'ORD-2026-0021'", engine)
print("New order:")
df

In [ ]:
# === UPDATE — อัปเดตข้อมูล ===
with engine.connect() as conn:
    conn.execute(text("""
        UPDATE test_orders 
        SET status = :new_status
        WHERE order_id = :oid
    """), {"new_status": "in_progress", "oid": "ORD-2026-0021"})
    conn.commit()

df = pd.read_sql("SELECT order_id, status FROM test_orders WHERE order_id = 'ORD-2026-0021'", engine)
print("Updated:")
df

In [ ]:
# === DELETE — ลบข้อมูล ===
with engine.connect() as conn:
    conn.execute(text("""
        DELETE FROM test_orders 
        WHERE order_id = :oid
    """), {"oid": "ORD-2026-0021"})
    conn.commit()

# ตรวจสอบว่าลบแล้ว
df = pd.read_sql("SELECT COUNT(*) AS total FROM test_orders", engine)
print(f"Total orders after delete: {df['total'][0]}")

## 6. Parameterized Queries — ป้องกัน SQL Injection ⚠️

**SQL Injection** = การโจมตีด้วยการส่ง SQL ที่อันตรายผ่าน input

```python
# ❌ อันตราย! ห้ามทำ!
query = f"SELECT * FROM users WHERE name = '{user_input}'"

# ✅ ปลอดภัย — ใช้ parameterized query
query = text("SELECT * FROM users WHERE name = :name")
conn.execute(query, {"name": user_input})
```

In [ ]:
# === ตัวอย่างการใช้ Parameterized Query ===

# สมมติ user เลือกค่าเหล่านี้จาก UI
selected_status = "completed"
selected_analyst = "Dr. Smith"

# ✅ ปลอดภัย
df = pd.read_sql(
    text("SELECT * FROM test_orders WHERE status = :s AND analyst_name = :a"),
    engine,
    params={"s": selected_status, "a": selected_analyst}
)

print(f"Found {len(df)} orders")
df.head()

## 7. Subquery และ CTE (Common Table Expression)

In [ ]:
# === Subquery — ค้นหา orders ที่มี As concentration สูงกว่าค่าเฉลี่ย ===
df = pd.read_sql("""
    SELECT DISTINCT
        o.order_id,
        o.sample_id,
        r.concentration AS as_concentration
    FROM test_results r
    INNER JOIN test_orders o ON r.order_id = o.id
    WHERE r.element_id = 1  -- Arsenic
      AND r.concentration > (
          SELECT AVG(concentration) 
          FROM test_results 
          WHERE element_id = 1
      )
    ORDER BY r.concentration DESC
""", engine)

print("Orders with above-average Arsenic:")
df.head()

In [ ]:
# === CTE — อ่านง่ายกว่า Subquery ===
df = pd.read_sql("""
    WITH avg_by_element AS (
        SELECT 
            element_id,
            AVG(concentration) AS avg_conc
        FROM test_results
        GROUP BY element_id
    )
    SELECT 
        e.symbol,
        e.name,
        ROUND(a.avg_conc::numeric, 3) AS avg_concentration,
        COUNT(r.id) AS high_count
    FROM test_results r
    INNER JOIN avg_by_element a ON r.element_id = a.element_id
    INNER JOIN elements e ON r.element_id = e.id
    WHERE r.concentration > a.avg_conc
    GROUP BY e.symbol, e.name, a.avg_conc
    ORDER BY high_count DESC
""", engine)

print("Count of above-average results per element:")
df

## 8. Window Functions — สำหรับ Ranking และ Running Totals

In [ ]:
# === ROW_NUMBER — ลำดับผลทดสอบ ===
df = pd.read_sql("""
    SELECT 
        o.order_id,
        e.symbol,
        r.concentration,
        ROW_NUMBER() OVER (
            PARTITION BY e.symbol 
            ORDER BY r.concentration DESC
        ) AS rank
    FROM test_results r
    INNER JOIN test_orders o ON r.order_id = o.id
    INNER JOIN elements e ON r.element_id = e.id
    WHERE r.replicate = 1
    ORDER BY e.symbol, rank
    LIMIT 16
""", engine)

print("Top results ranked by element:")
df

In [ ]:
# === Running Average — ค่าเฉลี่ยสะสม ===
df = pd.read_sql("""
    SELECT 
        o.order_id,
        o.order_date,
        r.concentration,
        ROUND(AVG(r.concentration) OVER (
            ORDER BY o.order_date
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        )::numeric, 3) AS moving_avg_3
    FROM test_results r
    INNER JOIN test_orders o ON r.order_id = o.id
    WHERE r.element_id = 1 AND r.replicate = 1
    ORDER BY o.order_date
""", engine)

print("Arsenic with 3-point moving average:")
df.head(10)

## 9. SQL + pandas — วิเคราะห์ต่อใน Python

In [ ]:
# ดึงข้อมูลทั้งหมดจาก DB
df_all = pd.read_sql("""
    SELECT 
        o.order_id,
        o.order_date,
        l.code AS lab,
        o.analyst_name,
        e.symbol AS element,
        r.concentration,
        r.replicate
    FROM test_results r
    INNER JOIN test_orders o ON r.order_id = o.id
    INNER JOIN elements e ON r.element_id = e.id
    INNER JOIN laboratories l ON o.laboratory_id = l.id
""", engine)

print(f"Total records: {len(df_all)}")
df_all.head()

In [ ]:
# วิเคราะห์ด้วย pandas
print("=== สรุปค่าเฉลี่ยตาม Lab และ Element ===")
pivot = df_all.pivot_table(
    values="concentration",
    index="lab",
    columns="element",
    aggfunc="mean"
).round(3)

pivot

In [ ]:
# Export สรุปเป็น Excel
pivot.to_excel("output/lab_summary_from_db.xlsx")
print("✅ Exported summary to output/lab_summary_from_db.xlsx")

## 📝 สรุป SQL Commands

| Command | ใช้ทำอะไร | ตัวอย่าง |
|---|---|---|
| `SELECT` | อ่านข้อมูล | `SELECT * FROM table` |
| `WHERE` | กรองข้อมูล | `WHERE status = 'completed'` |
| `JOIN` | รวมตาราง | `JOIN labs ON o.lab_id = l.id` |
| `GROUP BY` | จัดกลุ่ม | `GROUP BY status` |
| `HAVING` | กรองหลัง GROUP | `HAVING COUNT(*) > 5` |
| `ORDER BY` | เรียงลำดับ | `ORDER BY date DESC` |
| `INSERT` | เพิ่มข้อมูล | `INSERT INTO table VALUES (...)` |
| `UPDATE` | แก้ไขข้อมูล | `UPDATE table SET col = val` |
| `DELETE` | ลบข้อมูล | `DELETE FROM table WHERE ...` |

### Next: Notebook 17 — ETL Pipeline (Excel → JSON → Database)